In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os

In [12]:
current_dir = Path(os.path.dirname(os.path.abspath('__file__'))).parent
path = os.path.join(current_dir, "data/raw/ccnews/articles_filtered.jsonl")

# Read JSONL
df = pd.read_json(path, lines=True)

print("total unique urls:", df['url'].nunique())

lang = "ru"


# Keep lang only (adjust if you store different lang format)
df["lang"] = df["lang"].astype(str).str.lower().str.strip()
df_en = df[df["lang"]==lang].copy()

# Clean key fields
df_en["domain"] = df_en["domain"].astype(str).str.lower().str.strip()
df_en["url"] = df_en["url"].astype(str).str.strip()
df_en = df_en[(df_en["domain"] != lang) & (df_en["url"] != "")]

# Group by domain and count unique URLs
domain_stats = (
    df_en.groupby("domain", as_index=False)
    .agg(unique_urls=("url", "nunique"))
    .sort_values("unique_urls", ascending=False)
    .reset_index(drop=True)
)

# Share and cumulative share
total_unique_urls = domain_stats["unique_urls"].sum()
domain_stats["share_pct"] = domain_stats["unique_urls"] / total_unique_urls * 100
domain_stats["cum_share_pct"] = domain_stats["share_pct"].cumsum()

# Domains covering ~80%
top80 = domain_stats[domain_stats["cum_share_pct"] <= 80].copy()
if len(top80) < len(domain_stats):
    top80 = domain_stats.iloc[: len(top80) + 1].copy()

print(f"Total {lang} rows:", len(df_en))
print(f"Total {lang} unique URLs:", total_unique_urls)
print(f"Unique {lang} domains:", domain_stats["domain"].nunique())
print(f"Domains for ~80% coverage:", len(top80))
print(f"Coverage reached:", round(top80["share_pct"].sum(), 2), "%")

display(domain_stats.head(30))
display(top80)

KeyboardInterrupt: 

In [ ]:
# assumes df_en already exists and contains at least: date, url
tmp = df_en.copy()

# parse dates safely
tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
tmp = tmp[tmp["date"] > "2020-01-01"]
tmp = tmp.dropna(subset=["date", "url"])

# month bucket
tmp["month"] = tmp["date"].dt.to_period("M").dt.to_timestamp()

# monthly unique URLs
monthly_unique = (
    tmp.groupby("month")["url"]
    .nunique()
    .reset_index(name="unique_articles")
    .sort_values("month")
)

display(monthly_unique)

# plot
plt.figure(figsize=(10, 4))
plt.plot(monthly_unique["month"], monthly_unique["unique_articles"], marker="o")
plt.title("Monthly unique EN articles")
plt.xlabel("Month")
plt.ylabel("Unique articles")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# list of en domains
['www.aljazeera.com',
'www.channelnewsasia.com',
'www.thenationalnews.com',
'www.swissinfo.ch',
'www.straitstimes.com',
'vietnamnews.vn',
'abcnews.go.com',
'www.cbsnews.com',
'www.foxnews.com',
'www.latimes.com',
'globalnews.ca',
'www.ctvnews.ca',
'www.telegraph.co.uk',
'www.independent.co.uk',
'www.the-independent.com',
'www.irishtimes.com',
'www.scotsman.com',
'www.yorkshirepost.co.uk',
'timesofindia.indiatimes.com',
'economictimes.indiatimes.com',
'indianexpress.com',
'www.ndtv.com',
'www.business-standard.com',
'www.livemint.com',
'www.businesstoday.in',
'gulfnews.com',
'www.khaleejtimes.com',
'allafrica.com',
'www.timeslive.co.za',
'www.businesslive.co.za',
'thewest.com.au',
'www.nzherald.co.nz',
'www.nasdaq.com',
'seekingalpha.com',
'qz.com',
'www.barchart.com',
'www.pymnts.com']

In [ ]:
# ru domains
[
'tass.ru',
'ria.ru',
'www.interfax.ru',
'www.rbc.ru',
'www.vedomosti.ru',
'www.kommersant.ru',
'lenta.ru',
'www.nur.kz',
'www.zakon.kz'
]

## check progress

In [5]:
import polars as pl
from pathlib import Path
import os
from src.collectors.ccnews_extractor import load_processed, load_month_paths

current_dir = Path(os.path.dirname(os.path.abspath("__file__"))).parent
DATA_DIR = current_dir / "data/raw/ccnews"
PROCESSED_LOG = DATA_DIR / "processed_warcs_filtered.json"
BASE_URL = "https://data.commoncrawl.org/"
MONTHS = [f"{y}/{m:02d}" for y in [2025] for m in range(1, 13)] + ["2026/01", "2026/02", "2026/03"]

processed_set = set(load_processed(str(PROCESSED_LOG)))

progress_rows = []
parts = []

for month in MONTHS:
    year, mon = month.split("/")
    month_dir = DATA_DIR / year / mon
    files = sorted([*month_dir.glob("*.jsonl"), *month_dir.glob("*.parquet")]) if month_dir.exists() else []
    month_bytes = sum(f.stat().st_size for f in files)

    month_paths = load_month_paths(BASE_URL, month)
    total = len(month_paths)
    done = sum(p in processed_set for p in month_paths)
    left = total - done
    pct = round((done / total * 100) if total else 0.0, 2)

    month_rows = 0
    for f in files:
        if f.suffix == ".jsonl" and f.stat().st_size == 0:
            continue
        p = pl.read_parquet(f) if f.suffix == ".parquet" else pl.read_ndjson(f)
        p = p.with_columns(pl.lit(str(f)).alias("source_file"))
        month_rows += p.height
        parts.append(p)

    progress_rows.append({
        "month": month,
        "total_warcs": total,
        "processed_warcs": done,
        "remaining_warcs": left,
        "processed_pct": pct,
        "files_found": len(files),
        "rows_found": month_rows,
        "volume_mb": round(month_bytes / (1024 * 1024), 2),
    })

legacy_path = DATA_DIR / "articles_filtered.jsonl"
if legacy_path.exists() and legacy_path.stat().st_size > 0:
    parts.append(
        pl.read_ndjson(legacy_path).with_columns(pl.lit(str(legacy_path)).alias("source_file"))
    )

df = pl.concat(parts, how="diagonal_relaxed") if parts else pl.DataFrame()
if "url" in df.columns:
    df = df.unique(subset=["url"], keep="first")

progress_df = pl.DataFrame(progress_rows).sort("month")

year_total = int(progress_df["total_warcs"].sum())
year_done = int(progress_df["processed_warcs"].sum())
year_left = int(progress_df["remaining_warcs"].sum())
year_pct = round((year_done / year_total * 100) if year_total else 0.0, 2)

print(f"processed_log_entries={len(processed_set)}")
print(f"year_total_warcs={year_total}")
print(f"year_processed_warcs={year_done}")
print(f"year_remaining_warcs={year_left}")
print(f"year_processed_pct={year_pct}%")
print(f"files_found_total={int(progress_df['files_found'].sum())}")
print(f"rows_found_total={int(progress_df['rows_found'].sum())}")
print(f"df_rows_unique_urls={df.height}")
print(f"volume_total_mb={round(float(progress_df['volume_mb'].sum()), 2)}")

progress_df

processed_log_entries=2496
year_total_warcs=7235
year_processed_warcs=2496
year_remaining_warcs=4739
year_processed_pct=34.5%
files_found_total=2361
rows_found_total=513011
df_rows_unique_urls=768876
volume_total_mb=1778.74


month,total_warcs,processed_warcs,remaining_warcs,processed_pct,files_found,rows_found,volume_mb
str,i64,i64,i64,f64,i64,i64,f64
"""2025/01""",403,146,257,36.23,136,37952,134.1
"""2025/02""",386,116,270,30.05,101,27287,97.47
"""2025/03""",462,167,295,36.15,160,41653,144.85
"""2025/04""",491,176,315,35.85,171,39516,132.81
"""2025/05""",479,168,311,35.07,152,38325,129.96
…,…,…,…,…,…,…,…
"""2025/11""",574,194,380,33.8,185,29392,104.74
"""2025/12""",616,211,405,34.25,206,29491,105.5
"""2026/01""",483,154,329,31.88,142,28432,99.58


In [7]:
(
    df.group_by("domain")
      .agg(pl.col("url").n_unique().alias("unique_urls"))
      .sort("unique_urls", descending=True)
      .head(30)
)

domain,unique_urls
str,u32
"""www.kommersant.ru""",86940
"""www.the-independent.com""",57640
"""www.independent.co.uk""",49822
"""www.newindianexpress.com""",41679
"""www.cbsnews.com""",40424
…,…
"""www.nzherald.co.nz""",10568
"""www.yorkshirepost.co.uk""",10090
"""www.irishtimes.com""",9816
